In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

from scipy.ndimage import uniform_filter

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
#IMPORT CLASSES (from current directory)
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import * # import NumericalFunctions 

In [ ]:
####################################
#LOADING CLASSES

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=1)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="CalculateMeans",
                                dtype='float32')

In [ ]:
#JOB ARRAY SETUP
UsingJobArray=True

def GetNumJobs(res,t_res):
    if res=='1km':
        if t_res=='5min':
            num_jobs=20
        elif t_res=='3min':
            num_jobs=30
        elif t_res=='1min':
            num_jobs=100
    elif res=='250m': 
        if t_res=='1min':
            num_jobs=500
    return num_jobs
num_jobs = GetNumJobs(ModelData.res,ModelData.t_res)
SlurmJobArray = SlurmJobArray_Class(total_elements=ModelData.Ntime, num_jobs=num_jobs, UsingJobArray=UsingJobArray)
start_job = SlurmJobArray.start_job; end_job = SlurmJobArray.end_job

def GetNumElements():
    loop_elements = np.arange(ModelData.Ntime)[start_job:end_job]
    return loop_elements
loop_elements = GetNumElements()

In [ ]:
####################################
#FUNCTIONS

In [ ]:
#Calculation Functions

def GetVarNames():
    varNames=['qv','theta_v','theta_e','RH_vapor','HMC']
    return varNames
    
def GetInputVariables(inputDataDirectory, timeString):

    ################################# DENSITY POTENTIAL TEMPERATURE
    theta_rho = CallVariable(ModelData, DataManager, timeString, 'theta_v') #*yes, named theta_v, but really theta_rho after condensation

    varNames=GetVarNames()
    inputDictionary = {varName: CallVariable(ModelData, DataManager, timeString, varName) for varName in varNames}
    return inputDictionary

def VariableCalculation(inputDictionary):

    outputDictionary = {}
    for varName, variableData in inputDictionary.items():
        outputDictionary[f"{varName}_mean"] = np.nanmean(variableData,axis=(1, 2),keepdims=False)
    return outputDictionary

In [ ]:
####################################
#RUNNING
running = False
# running = True

In [ ]:
if running:
    #CALCULATING AND APPENDING TO DATA EACH TIMESTEP
    for t in tqdm(loop_elements, total=len(loop_elements)):
        if np.mod(t,1)==0: print(f'Current time {t}')
    
        #getting timestring for loading input data
        timeString = ModelData.timeStrings[t]
    
        #loading input variables
        inputDictionary = GetInputVariables(DataManager.inputDataDirectory, timeString)
    
        #calculating variables
        outputDictionary = VariableCalculation(inputDictionary)
        
        #outputting
        DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, timeString, outputDictionary)

In [ ]:
####################################
#RECOMBINING

In [ ]:
def Recombine(ModelData,DataManager):
    # Pre-allocate arrays of zeros
    varNames=GetVarNames()
    varNames=[varName+"_mean" for varName in varNames]
    combinedDictionary = {varName.lower(): np.zeros((ModelData.Ntime, ModelData.Nzh)) for varName in varNames}
    
    # Fill in all timesteps
    for t in tqdm(range(ModelData.Ntime)):
    
        #loading data
        outputDictionary = {varName: CallVariable(ModelData, DataManager, ModelData.timeStrings[t], varName) for varName in varNames}
        #combining data
        for varName, variableData in outputDictionary.items(): combinedDictionary[varName.lower()][t] = variableData
    
    #saving
    DataManager.SaveOutputTimestep(DataManager.outputDataDirectory, "all_times", combinedDictionary)

In [ ]:
if not running:
    Recombine(ModelData,DataManager)

In [ ]:
####################################
#READING BACK IN

In [ ]:
# DataManager_CalculateMeans = DataManager_Class(mainDirectory, scratchDirectory, ModelData, dataType="CalculateMoreVariables", dataName="CalculateMeans",
#                                 dtype='float32')

# [inputDataDirectory,dataName] = CallVariable(ModelData, DataManager_CalculateMeans, 'all_times', 'qv_mean',loadData=False) 
# meanData = DataManager.GetTimestepData_V2(inputDataDirectory, 'all_times', dataName=dataName)
# meanData[varName][:]
# # ...
# meanData.close()

In [ ]:
####################################
#TESTING

In [ ]:
# #testing plotting
# def TestPlot():
#     fig,axis=plt.subplots()
#     a=combinedDictionary['qv_mean']
#     plt.contourf(np.arange(ModelData.Ntime),ModelData.zh,a.T,cmap='turbo',levels=20);plt.colorbar()
#     fig,axis=plt.subplots()
#     a=combinedDictionary['theta_v_mean']
#     plt.contourf(np.arange(ModelData.Ntime),ModelData.zh,a.T,cmap='turbo',levels=100);plt.colorbar()
# TestPlot()